In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [ ]:
# 1. Load and prepare data

In [8]:
def load_loan_data(csv_path):
    """
    Load loan dataset.
    Assumes columns like:
    income, loan_amount, total_loans_outstanding, age, previous_default, default
    """
    df = pd.read_csv(r"Task 3 and 4_Loan_Data.csv")
    return df

In [ ]:
# 2. Train PD model

In [9]:
def train_pd_model(df):
    """
    Train a logistic regression model to estimate PD.
    """
    # Features (customize based on dataset)
    feature_cols = [col for col in df.columns if col not in ["default"]]
    X = df[feature_cols]
    y = df["default"]

    # Train-test split (for validation)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Pipeline: scaling + logistic regression
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=1000))
    ])

    model.fit(X_train, y_train)

    print("Training accuracy:", model.score(X_train, y_train))
    print("Test accuracy:", model.score(X_test, y_test))

    return model, feature_cols

In [ ]:
# 3. Expected Loss function

In [16]:
def expected_loss(model, feature_cols, borrower_features, recovery_rate=0.10):
    # Convert borrower dict to DataFrame
    X = pd.DataFrame([borrower_features])

    # Add any missing columns with default value 0
    for col in feature_cols:
        if col not in X.columns:
            X[col] = 0

    # Reorder columns to match training
    X = X[feature_cols]

    # Predict PD
    pd_value = model.predict_proba(X)[0][1]

    # LGD = 1 - recovery rate
    lgd = 1 - recovery_rate

    # EAD = loan amount outstanding
    ead = borrower_features.get("loan_amt_outstanding", 0)

    # Expected Loss
    el = pd_value * lgd * ead

    return {
        "PD": pd_value,
        "LGD": lgd,
        "EAD": ead,
        "Expected_Loss": el
    }


In [17]:
print(feature_cols)

['customer_id', 'credit_lines_outstanding', 'loan_amt_outstanding', 'total_debt_outstanding', 'income', 'years_employed', 'fico_score']


In [18]:
# 4. Example usage

df = load_loan_data("Task 3 and 4_Loan_Data.csv")
model, feature_cols = train_pd_model(df)

borrower = {
    "customer_id": 12345,
    "credit_lines_outstanding": 3,
    "loan_amt_outstanding": 15000,
    "total_debt_outstanding": 3000,
    "income": 60000,
    "years_employed": 5,
    "fico_score": 720
}

result = expected_loss(model, feature_cols, borrower)
print(result)

Training accuracy: 0.99825
Test accuracy: 0.9955
{'PD': np.float64(0.00011684191575305852), 'LGD': 0.9, 'EAD': 15000, 'Expected_Loss': np.float64(1.57736586266629)}
